# The Sum of Gaussians

Everything from this lesson — expectations, linearity, variance, standard deviation — comes together in one neat application: **adding two Gaussian random variables**. If a computer system's response time is processing time plus network latency, and both are Gaussian and independent, the total is *still Gaussian* — with mean the sum of the means, and **variance** (not standard deviation!) the sum of the variances. It is based on the seventh video of Lesson 3 (Describing Distributions).

## 1. Introduction

Imagine you are studying the **total response time** of a computer system. It consists of two components:

- the **processing time** $T$ — the time the system takes to process a task;
- the **network latency** $L$ — the communication delay between the system and other devices (database servers, external APIs).

The total response time is their sum: $R = T + L$. If you can model each component, what can you say about the total? This is the everyday shape of real problems: totals, differences, and weighted combinations of random quantities.

After completing this notebook you will be able to:

- State what happens when you add independent Gaussian variables (the result stays Gaussian).
- Compute the parameters of the sum: $\mu_R = \mu_T + \mu_L$ and $\sigma_R^2 = \sigma_T^2 + \sigma_L^2$.
- Explain where independence is needed (the variance step) and where it is not (the mean step).
- Use the general rule for a linear combination $W = aX + bY$ of independent Gaussians.

## 2. Theory

### The setup

Model both components (in milliseconds) as Gaussian, and assume they are **independent**:

$$
T \sim N(\mu_T = 10,\ \sigma_T = 2), \qquad L \sim N(\mu_L = 5,\ \sigma_L = 1)
$$

Sample each 10,000 times: both histograms fit their bell curves well. Now add each pair of samples to get 10,000 samples of $R = T + L$ and draw *that* histogram. Two things stand out:

1. **$R$ is still Gaussian.** The sum of independent Gaussian variables is itself Gaussian — the bell shape survives addition. So $R$ is fully described by just two numbers: its $\mu_R$ and $\sigma_R$.
2. The new bell is centered further right and slightly wider — let's compute exactly where and how wide.

### The mean of the sum

$$
\mu_R = E[R] = E[T + L] = E[T] + E[L] = \mu_T + \mu_L = 10 + 5 = 15
$$

This step uses only the **linearity of expectation** — it works for *any* two variables, independent or not.

### The variance of the sum

$$
\sigma_R^2 = \mathrm{Var}(R) = \mathrm{Var}(T + L) = \mathrm{Var}(T) + \mathrm{Var}(L) = \sigma_T^2 + \sigma_L^2 = 4 + 1 = 5
$$

**Careful — this is not a general result.** Variances add only because $T$ and $L$ are **independent**. (Dependent variables can reinforce or cancel each other's fluctuations, inflating or shrinking the variance of the sum.)

So the standard deviation of the total is

$$
\sigma_R = \sqrt{\sigma_T^2 + \sigma_L^2} = \sqrt{5} \approx 2.24
$$

Note the shape of this formula — a *square root of a sum of squares*, like a Pythagorean theorem for spreads. **Standard deviations do not add**: $\sigma_R \approx 2.24$, not $2 + 1 = 3$. It is the variances that add.

### The general rule: linear combinations

For any constants $a, b$ and **independent** Gaussians

$$
X \sim N(\mu_X, \sigma_X^2), \qquad Y \sim N(\mu_Y, \sigma_Y^2),
$$

the linear combination $W = aX + bY$ is Gaussian with

$$
W \sim N\big(a\mu_X + b\mu_Y,\ a^2\sigma_X^2 + b^2\sigma_Y^2\big)
$$

Both ingredients are familiar from this lesson: the mean comes from linearity of expectation, and the variance combines $\mathrm{Var}(aX) = a^2\mathrm{Var}(X)$ with the independence rule for sums. Note $b^2$ in the variance: even a *difference* ($b = -1$) has the variances **adding** — subtracting a noisy quantity adds noise, never removes it.

## 3. Worked Examples

### Example 1: the response time

$T \sim N(10, 2^2)$, $L \sim N(5, 1^2)$, independent:

$$
R = T + L \sim N\big(15,\ 5\big), \qquad \sigma_R = \sqrt{5} \approx 2.24 \text{ ms}
$$

### Example 2: a difference of Gaussians

How much *longer* is processing than latency? $D = T - L$ is the case $a = 1$, $b = -1$:

$$
\mu_D = 10 - 5 = 5, \qquad \sigma_D^2 = 1^2\cdot 4 + (-1)^2 \cdot 1 = 5
$$

So $D \sim N(5, 5)$ — same variance as the sum! Subtracting an independent noisy variable adds just as much spread as adding it.

### Example 3: averaging two independent measurements

Two independent instruments measure the same length, each with error distribution $N(\mu, \sigma^2)$. Their average $W = \tfrac12 X + \tfrac12 Y$ has

$$
\mu_W = \tfrac12\mu + \tfrac12\mu = \mu, \qquad
\sigma_W^2 = \tfrac14\sigma^2 + \tfrac14\sigma^2 = \tfrac{\sigma^2}{2}
$$

Same center, **half the variance** ($\sigma$ shrinks by $\sqrt2$): averaging independent measurements is *more precise* than either measurement alone — the statistical reason for repeating experiments.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

def normal_pdf(x, mu, sigma):
    return np.exp(-0.5 * ((x - mu) / sigma)**2) / (sigma * np.sqrt(2 * np.pi))

### Reproducing the video's experiment: sample, sum, inspect

In [ ]:
rng = np.random.default_rng(42)

n_samples = 10_000
processing = rng.normal(10, 2, size=n_samples)     # T ~ N(10, 2^2)
latency = rng.normal(5, 1, size=n_samples)         # L ~ N(5, 1^2)
response = processing + latency                    # R = T + L

print(f"T: sample mean = {processing.mean():.3f}, sample std = {processing.std():.3f}")
print(f"L: sample mean = {latency.mean():.3f}, sample std = {latency.std():.3f}")
print(f"R: sample mean = {response.mean():.3f}, sample std = {response.std():.3f}")
print(f"\nTheory for R: mu = 10 + 5 = 15, sigma = sqrt(4 + 1) = {np.sqrt(5):.3f}")
print(f"Note: standard deviations do NOT add (2 + 1 = 3 is wrong).")

### The general rule $W = aX + bY$ for a few combinations

In [ ]:
rng = np.random.default_rng(42)

n = 1_000_000
t = rng.normal(10, 2, size=n)
l = rng.normal(5, 1, size=n)

cases = [("R = T + L", 1, 1), ("D = T - L", 1, -1), ("W = 3T + 2L", 3, 2)]
print(f"{'combination':<12} {'sim mean':>9} {'theory':>7} {'sim std':>9} {'theory':>7}")
for name, a, b in cases:
    w = a * t + b * l
    mu_theory = a * 10 + b * 5
    sigma_theory = np.sqrt(a**2 * 4 + b**2 * 1)
    print(f"{name:<12} {w.mean():>9.3f} {mu_theory:>7.1f} {w.std():>9.3f} {sigma_theory:>7.3f}")

### Where independence matters

Make the latency *depend* on the processing time (e.g. heavy tasks also clog the network): the mean of the sum is still 15, but the variance is no longer $4 + 1$.

In [ ]:
rng = np.random.default_rng(42)

n = 1_000_000
t = rng.normal(10, 2, size=n)

# Independent latency vs latency that moves with T (same N(5,1) marginal!)
l_indep = rng.normal(5, 1, size=n)
l_dep = 5 + (t - 10) / 2                     # perfectly correlated with T, std = 1

for name, l in [("independent", l_indep), ("dependent  ", l_dep)]:
    r = t + l
    print(f"L {name}: E[R] = {r.mean():.3f}, Var(R) = {r.var():.3f}"
          f"   (independent theory: 15, 5)")
print("\nDependence leaves the mean rule intact but breaks Var(T+L) = Var(T)+Var(L).")

## 5. Visualizations

### The two components and their sum

The video's three plots in one figure: histograms of $T$, $L$, and $R = T + L$, each with its Gaussian density — the sum is still a bell, centered at 15 with $\sigma = \sqrt5$.

In [ ]:
rng = np.random.default_rng(42)

n_samples = 10_000
t = rng.normal(10, 2, size=n_samples)
l = rng.normal(5, 1, size=n_samples)
r = t + l

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)
for ax, data, mu, sigma, title, color in [
        (axes[0], t, 10, 2, "processing time  T ~ N(10, 2²)", "tab:blue"),
        (axes[1], l, 5, 1, "network latency  L ~ N(5, 1²)", "tab:orange"),
        (axes[2], r, 15, np.sqrt(5), "response time  R = T + L ~ N(15, 5)", "tab:green")]:
    x = np.linspace(mu - 4 * sigma, mu + 4 * sigma, 400)
    ax.hist(data, bins=50, density=True, color=color, alpha=0.55,
            edgecolor="white", label="10,000 samples")
    ax.plot(x, normal_pdf(x, mu, sigma), color="black", linewidth=2,
            label=f"N({mu}, {sigma**2:.0f})")
    ax.set_title(title)
    ax.set_xlabel("milliseconds")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)
axes[0].set_ylabel("density")
plt.suptitle("Adding independent Gaussians: the sum is still Gaussian")
plt.tight_layout()
plt.show()

### Standard deviations do not add — variances do

If standard deviations added, the sum's bell (dashed red) would be noticeably wider than the real one. The Pythagorean combination $\sqrt{2^2 + 1^2}$ matches the data.

In [ ]:
rng = np.random.default_rng(42)
r = rng.normal(10, 2, size=100_000) + rng.normal(5, 1, size=100_000)

x = np.linspace(4, 26, 500)

plt.figure(figsize=(9, 5))
plt.hist(r, bins=80, density=True, color="tab:green", alpha=0.5,
         edgecolor="white", label="samples of R = T + L")
plt.plot(x, normal_pdf(x, 15, np.sqrt(5)), color="black", linewidth=2,
         label=r"correct: $\sigma_R = \sqrt{2^2 + 1^2} = \sqrt{5} \approx 2.24$")
plt.plot(x, normal_pdf(x, 15, 3), color="tab:red", linewidth=2, linestyle="--",
         label=r"wrong: $\sigma_R = 2 + 1 = 3$")
plt.title("The spread of a sum combines like Pythagoras, not like addition")
plt.xlabel("response time (ms)")
plt.ylabel("density")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### Averaging reduces spread

The average of two independent copies of $N(10, 2^2)$ keeps the center but shrinks $\sigma$ by $\sqrt2$ — the width payoff of repeated measurements.

In [ ]:
rng = np.random.default_rng(42)

x1 = rng.normal(10, 2, size=100_000)
x2 = rng.normal(10, 2, size=100_000)
avg = (x1 + x2) / 2

x = np.linspace(2, 18, 500)
plt.figure(figsize=(9, 5))
plt.hist(x1, bins=80, density=True, color="tab:blue", alpha=0.4,
         edgecolor="white", label="one measurement  N(10, 2²)")
plt.hist(avg, bins=80, density=True, color="tab:purple", alpha=0.5,
         edgecolor="white", label="average of two  N(10, 2²/2)")
plt.plot(x, normal_pdf(x, 10, 2), color="tab:blue", linewidth=2)
plt.plot(x, normal_pdf(x, 10, 2 / np.sqrt(2)), color="tab:purple", linewidth=2)
plt.title(r"$W = \frac{1}{2}X + \frac{1}{2}Y$: same mean, $\sigma$ divided by $\sqrt{2}$")
plt.xlabel("measured value")
plt.ylabel("density")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### The whole story on one page

Everything above in a single sheet: the response-time sum with its simulation, why variances (not standard deviations) add, differences, and averaging.

<img src="img/07_sum_of_gaussians.png" alt="Concept sheet for this notebook" width="900">

## 6. Exercises

### Basic

**Exercise 1.** $X \sim N(3, 2^2)$ and $Y \sim N(7, 3^2)$ are independent. What are the distribution, mean, and standard deviation of $S = X + Y$?

**Exercise 2.** For the same $X$ and $Y$, find the distribution of $D = Y - X$. Compare $\sigma_D$ with $\sigma_S$.

### Intermediate

**Exercise 3.** A delivery consists of preparation time $\sim N(20, 4^2)$ minutes and travel time $\sim N(30, 3^2)$ minutes, independent. Using the 68–95–99.7 rule on the total, give a range that covers about 95% of deliveries.

**Exercise 4.** Two independent scales weigh the same object: readings $\sim N(\mu, 10^2)$ grams each. Compare three estimates of the weight — scale 1 alone, the average of both, and the weighted combination $0.8 X_1 + 0.2 X_2$ — by computing each estimator's standard deviation. Which is most precise?

### Advanced

**Exercise 5.** Generalize Example 3: the average of $n$ independent copies of $N(\mu, \sigma^2)$ is $N(\mu, \sigma^2/n)$, so its standard deviation is $\sigma/\sqrt{n}$. Verify by simulation for $n = 1, 4, 16, 64$ — each quadrupling of $n$ should halve the spread.

**Exercise 6.** The Gaussian is special: sums of independent Gaussians are exactly Gaussian, but sums of other distributions are not (immediately). Simulate the sum of 2 independent uniform variables on $[0,1]$ (triangle!), then the sum of 20. What do you observe, and which famous theorem is peeking through?

## 7. Solutions

In [ ]:
# Exercise 1: sum of independent Gaussians
mu_s = 3 + 7
var_s = 2**2 + 3**2
print(f"Exercise 1: S ~ N({mu_s}, {var_s}), sigma_S = sqrt({var_s}) = {np.sqrt(var_s):.3f}")

In [ ]:
# Exercise 2: difference
mu_d = 7 - 3
var_d = (-1)**2 * 2**2 + 1**2 * 3**2
print(f"Exercise 2: D ~ N({mu_d}, {var_d}), sigma_D = {np.sqrt(var_d):.3f}")
print("Same spread as the sum: subtracting independent noise adds variance too.")

In [ ]:
# Exercise 3: delivery times
mu = 20 + 30
sigma = np.sqrt(4**2 + 3**2)
print(f"Exercise 3: total ~ N({mu}, {sigma**2:.0f}), sigma = {sigma} min")
print(f"95% of deliveries in mu +- 2 sigma = [{mu - 2*sigma:.0f}, {mu + 2*sigma:.0f}] minutes")

In [ ]:
# Exercise 4: combining two scales
combos = [("scale 1 alone      ", 1.0, 0.0),
          ("average (0.5, 0.5) ", 0.5, 0.5),
          ("weighted (0.8, 0.2)", 0.8, 0.2)]
print("Exercise 4:")
for name, a, b in combos:
    sigma = np.sqrt(a**2 * 100 + b**2 * 100)
    print(f"  {name}: sigma = {sigma:.2f} g")
print("The 50/50 average is the most precise unbiased combination of two")
print("equally good instruments (sigma 7.07 < 8.25 < 10).")

In [ ]:
# Exercise 5: sigma / sqrt(n)
rng = np.random.default_rng(0)

mu, sigma = 10, 2
print("Exercise 5:")
print(f"{'n':>4} {'sim std':>9} {'sigma/sqrt(n)':>14}")
for n in [1, 4, 16, 64]:
    averages = rng.normal(mu, sigma, size=(200_000, n)).mean(axis=1)
    print(f"{n:>4} {averages.std():>9.4f} {sigma/np.sqrt(n):>14.4f}")
print("Each 4x increase in n halves the spread of the average.")

In [ ]:
# Exercise 6: sums of uniforms
rng = np.random.default_rng(0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, k in [(axes[0], 2), (axes[1], 20)]:
    sums = rng.random(size=(200_000, k)).sum(axis=1)
    ax.hist(sums, bins=80, density=True, color="tab:blue", alpha=0.6,
            edgecolor="white", label=f"sum of {k} uniforms")
    mu_k, sigma_k = k * 0.5, np.sqrt(k / 12)
    x = np.linspace(mu_k - 4 * sigma_k, mu_k + 4 * sigma_k, 300)
    ax.plot(x, normal_pdf(x, mu_k, sigma_k), color="black", linewidth=2,
            label="Gaussian with same mu, sigma")
    ax.set_title(f"sum of {k} independent U[0,1]")
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

print("Exercise 6: two uniforms give a TRIANGLE, not a bell -- the 'stays the same")
print("family' property is special to Gaussians. But by 20 terms the sum is already")
print("almost perfectly bell-shaped: the Central Limit Theorem at work.")

## 8. Conclusion

- For **independent Gaussians**, the sum is still Gaussian: $R = T + L \sim N(15, 5)$ for $T \sim N(10, 4)$ and $L \sim N(5, 1)$.
- The **mean** of the sum needs only linearity — $\mu_R = \mu_T + \mu_L$, independence or not.
- The **variance** of the sum adds only under **independence**: $\sigma_R^2 = \sigma_T^2 + \sigma_L^2$. Standard deviations combine like Pythagoras: $\sigma_R = \sqrt{\sigma_T^2 + \sigma_L^2} = \sqrt5 \approx 2.24$, not $2 + 1$.
- General rule: for independent $X, Y$ Gaussian and constants $a, b$: $\ aX + bY \sim N(a\mu_X + b\mu_Y,\ a^2\sigma_X^2 + b^2\sigma_Y^2)$ — note the squares: even differences get *wider*, and averages get *narrower* ($\sigma/\sqrt{n}$ for $n$ measurements).
- This little calculation is the workhorse behind error propagation, signal processing, and (via the Central Limit Theorem, peeking through in the exercises) much of statistics.